In [1]:
import tensorflow as tf
import os
import glob
import numpy as np
from model import build_1d_cnn_model
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES, fast_gpu_map,parse_example
cnn_model=build_1d_cnn_model(None,INPUT_SHAPE,OUTPUT_DIM_NOTES,False)
cnn_model.summary()
tf.keras.utils.plot_model(cnn_model,to_file='cnn_model.png',show_shapes=True)
# cnn_model.load_weights('/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/checkpoints/guitarmidi_epoch02_valAcc0.9551.weights.h5')#
cnn_model.load_weights('guitarmidi.weights.h5')

input_data_dir = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/data_slices'
input_filepaths = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/data_slices/training_subset/filtered_poly_data.tfrecord'#sorted(glob.glob(os.path.join(input_data_dir, '**', 'input', 'data.tfrecord'), recursive=True))
# train_dataset = tf.data.Dataset.from_tensor_slices((input_filepaths))
# train_dataset=train_dataset.shuffle(buffer_size=len(input_filepaths))
# train_dataset=train_dataset.take(100)
# train_dataset = train_dataset.map(tf_load_sample_from_files, num_parallel_calls=tf.data.AUTOTUNE)
def representative_data_gen():
    # Use TFRecordDataset to actually read the files
    # We only need a few samples to calibrate quantization
    raw_dataset = tf.data.TFRecordDataset(input_filepaths).map(parse_example).take(100)
    
    # Map using your existing loading function
    calib_dataset = raw_dataset.map(lambda path: fast_gpu_map(path, training=False))
    
    for input_value, _ in calib_dataset.batch(1):
        # input_value is the (312, 256, 1) tensor
        yield [input_value]

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)

converter.optimizations = [ tf.lite.Optimize.DEFAULT]
# converter.representative_dataset = representative_data_gen
# converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
# converter.target_spec.supported_types = [tf.int8] 

# converter.target_spec.supported_types = [tf.float16]
# converter.target_spec._experimental_supported_accumulation_type = tf.dtypes.float16
# Perform the conversion
tflite_model = converter.convert()


# converter=tf.lite.TFLiteConverter.from_keras_model(cnn_model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# tflite_model=converter.convert()

with open('guitarmidi.tflite','wb') as f:
    f.write(tflite_model)
print("TFLite model saved as guitarmidi.tflite")

I0000 00:00:1773655528.069390  882841 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773655528.099846  882841 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773655528.573356  882841 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1773655529.170786  882841 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel bi

Initial input shape: (None, 148, 512)
After first Conv2D: (None, 74, 64)
Extracting string from filters 0 to 26
String 0 section shape: (None, 26, 64)
String 0 after first Conv1D: (None, 26, 128)
Extracting string from filters 11 to 36
String 11 section shape: (None, 25, 64)
String 11 after first Conv1D: (None, 25, 128)
Extracting string from filters 21 to 46
String 21 section shape: (None, 25, 64)
String 21 after first Conv1D: (None, 25, 128)
Extracting string from filters 31 to 56
String 31 section shape: (None, 25, 64)
String 31 after first Conv1D: (None, 25, 128)
Extracting string from filters 39 to 64
String 39 section shape: (None, 25, 64)
String 39 after first Conv1D: (None, 25, 128)
Extracting string from filters 49 to 73
String 49 section shape: (None, 24, 64)
String 49 after first Conv1D: (None, 24, 128)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 148, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 148, 256,  │          0 │ input_layer[0][0] │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 148, 32,   │        272 │ reshape[0][0]     │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 148, 32,   │         64 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu         │ (None, 148, 32,   │          0 │ batch_normalizat… │
│ (LeakyReLU)         │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout2d   │ (None, 148, 32,   │          0 │ leaky_re_lu[0][0] │
│ (SpatialDropout2D)  │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 148, 512)  │          0 │ spatial_dropout2… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 148, 512)  │    263,040 │ reshape_1[0][0],  │
│ (MultiHeadAttentio… │                   │            │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 148, 512)  │          0 │ reshape_1[0][0],  │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 148, 512)  │      1,024 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 148, 256)  │    131,328 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 148, 512)  │    131,584 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 148, 512)  │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 148, 512)  │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 148, 512)  │      1,024 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 148, 32)   │    114,720 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 148, 32)   │        128 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_1       │ (None, 148, 32)   │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 4,553,244 (17.37 MB)

...

 Non-trainable params: 4,896 (19.12 KB)

INFO:tensorflow:Assets written to: /tmp/tmpi8kfe__d/assets


INFO:tensorflow:Assets written to: /tmp/tmpi8kfe__d/assets


Saved artifact at '/tmp/tmpi8kfe__d'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 148, 256, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 44), dtype=tf.float32, name=None)
Captures:
  139267552153424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139267552154576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139267552152272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139267552152656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139267552154000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139267552154384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139267552155152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139267552156112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139267552155920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139267552156496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1392675521563

W0000 00:00:1773655533.479241  882841 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1773655533.479262  882841 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1773655533.480077  882841 reader.cc:83] Reading SavedModel from: /tmp/tmpi8kfe__d
I0000 00:00:1773655533.481686  882841 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1773655533.481694  882841 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpi8kfe__d
I0000 00:00:1773655533.498850  882841 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1773655533.502712  882841 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1773655533.635531  882841 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpi8kfe__d
I0000 00:00:1773655533.667177  882841 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 187110 microseconds.
I0000 00:00:1773655533.716153  88284